In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

In [5]:
import requests

r = requests.get(
    "https://api.metalpriceapi.com/v1/latest",
    params={"api_key": "41197866644dfb3a514e2fe78a8b968b", "base": "USD", "currencies": "XPT,XPD"}
)

{'USDXPD': 1492.070762948, 'USDXPT': 1971.9494139761, 'XPD': 0.0006702095, 'XPT': 0.0005071124}


In [7]:
platinium = r.json()['rates']['USDXPT']
palladium = r.json()['rates']['USDXPD']


1971.9494139761


In [8]:
import requests
import pandas as pd
from datetime import date, timedelta

API_KEY = "YOUR_API_KEY"
BASE_URL = "https://api-eu.metalpriceapi.com/v1/timeframe"
METALS = ["XPT", "XPD", "XRH"]


def fetch_range(start: date, end: date) -> pd.DataFrame:
    """Fetch metal prices for a date range (max 365 days per call)."""
    params = {
        "api_key": '41197866644dfb3a514e2fe78a8b968b',
        "base": "USD",
        "currencies": ",".join(f"USD{m}" for m in METALS),
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
    }
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    data = r.json()

    if not data.get("success"):
        raise RuntimeError(data)

    # data["rates"] is a dict: {"2024-01-01": {"USDXPT": 912.3, ...}, ...}
    rows = {
        dt: {metal: day_rates.get(f"USD{metal}") for metal in METALS}
        for dt, day_rates in data["rates"].items()
    }
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
    return df


def fetch_history(start: date, end: date) -> pd.DataFrame:
    """Tile 365-day windows to cover arbitrarily long date ranges."""
    chunks = []
    cursor = start
    while cursor <= end:
        window_end = min(cursor + timedelta(days=364), end)
        chunks.append(fetch_range(cursor, window_end))
        cursor = window_end + timedelta(days=1)
    return pd.concat(chunks)


# Example: 3 years of data
df = fetch_history(date(2022, 1, 1), date(2024, 12, 31))
print(df.head())
#             XPT     XPD     XRH
# 2022-01-03  971.5  1911.0  14200.0
# ...

RuntimeError: {'success': False, 'error': {'statusCode': 412, 'message': 'Timeframe queries with multiple currencies require a paid plan. Please use a single currency or upgrade your plan.'}}